###Building the tables

In [0]:
from pyspark.sql import functions as F

def build(name, select_sql):
    spark.sql(f"CREATE OR REPLACE TABLE zomato_ai.silver.{name} AS {select_sql}")
    print("OK", name, spark.table(f"zomato_ai.silver.{name}").count(), "rows")

In [0]:
build("food", """
SELECT
  f_id AS food_id,
  trim(item) AS item,
  lower(trim(veg_or_non_veg)) AS veg_or_non_veg
FROM zomato_ai.bronze.food
WHERE f_id IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY f_id ORDER BY _ingested_at DESC) = 1
""")

OK food 371563 rows


In [0]:
build("users", """
SELECT
  user_id,
  trim(name) AS name,
  sha2(lower(trim(email)), 256) AS email_hash,
  try_cast(age AS int) AS age,
  lower(trim(gender)) AS gender,
  trim(marital_status) AS marital_status,
  trim(occupation) AS occupation,
  trim(monthly_income) AS monthly_income,
  trim(educational_qualifications) AS educational_qualifications,
  try_cast(family_size AS int) AS family_size
FROM zomato_ai.bronze.users
WHERE user_id IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY user_id ORDER BY _ingested_at DESC) = 1
""")

OK users 100000 rows


In [0]:
build("restaurants", """
SELECT
  id AS restaurant_id,
  trim(name) AS name,
  trim(city) AS city,
  try_cast(rating AS double) AS rating,
  try_cast(regexp_extract(rating_count, '(\\\\d+)', 1) AS int) AS rating_count,
  try_cast(regexp_extract(cost, '(\\\\d+)', 1) AS int) AS cost,
  trim(cuisine) AS cuisine,
  lic_no,
  trim(address) AS address
FROM zomato_ai.bronze.restaurants
WHERE id IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY id ORDER BY _ingested_at DESC) = 1
""")

OK restaurants 148544 rows


In [0]:
build("reviews", """
SELECT
  review_id,
  order_id,
  user_id,
  restaurant_id,
  try_cast(rating AS double) AS rating,
  trim(comment) AS comment,
  try_cast(review_date AS date) AS review_date
FROM zomato_ai.bronze.reviews
WHERE review_id IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY review_id ORDER BY _ingested_at DESC) = 1
""")

OK reviews 300000 rows


In [0]:
build("menu", """
SELECT DISTINCT
  menu_id,
  r_id AS restaurant_id,
  f_id AS food_id,
  trim(cuisine) AS cuisine,
  try_cast(price AS decimal(14,2)) AS price
FROM zomato_ai.bronze.menu
WHERE menu_id IS NOT NULL
""")

OK menu 1179936 rows


In [0]:
build("orders", """
SELECT
  order_id,
  try_cast(order_timestamp AS timestamp) AS order_timestamp,
  try_cast(order_date AS date) AS order_date,
  user_id,
  r_id AS restaurant_id,
  trim(restaurant_city) AS restaurant_city,
  trim(cuisine) AS cuisine,
  try_cast(items_count AS int) AS items_count,
  try_cast(sales_qty AS int) AS sales_qty,
  try_cast(subtotal AS decimal(14,2)) AS subtotal,
  try_cast(discount AS decimal(14,2)) AS discount,
  try_cast(delivery_fee AS decimal(14,2)) AS delivery_fee,
  try_cast(gst AS decimal(14,2)) AS gst,
  try_cast(sales_amount AS decimal(14,2)) AS sales_amount,
  currency,
  lower(trim(payment_method)) AS payment_method,
  lower(trim(order_status)) AS order_status,
  try_cast(customer_rating AS double) AS customer_rating,
  try_cast(try_cast(delivery_time_min AS double) AS int) AS delivery_time_min
FROM zomato_ai.bronze.orders
WHERE order_id IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY order_id ORDER BY _ingested_at DESC) = 1
""")

OK orders 10000000 rows


In [0]:
build("order_items", """
SELECT
  order_item_id,
  order_id,
  r_id AS restaurant_id,
  f_id AS food_id,
  try_cast(price AS decimal(14,2)) AS price,
  try_cast(quantity AS int) AS quantity,
  try_cast(line_amount AS decimal(14,2)) AS line_amount
FROM zomato_ai.bronze.order_items
WHERE order_item_id IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY order_item_id ORDER BY _ingested_at DESC) = 1
""")

OK order_items 22998179 rows


##Checks

###Seeing if only true duplicates were removed

In [0]:
for t in ["orders", "restaurants", "reviews", "users", "food", "menu", "order_items"]:
    b = spark.table(f"zomato_ai.bronze.{t}").count()
    s = spark.table(f"zomato_ai.silver.{t}").count()
    print(f"{t}: bronze {b} -> silver {s} (removed {b - s})")

orders: bronze 10000000 -> silver 10000000 (removed 0)
restaurants: bronze 148544 -> silver 148544 (removed 0)
reviews: bronze 300000 -> silver 300000 (removed 0)
users: bronze 100000 -> silver 100000 (removed 0)
food: bronze 371632 -> silver 371563 (removed 69)
menu: bronze 1179936 -> silver 1179936 (removed 0)
order_items: bronze 22998179 -> silver 22998179 (removed 0)


###A high null count would mean a type conversion failed.

In [0]:
checks = {
    "orders": ["order_timestamp", "order_date", "subtotal", "sales_amount", "delivery_time_min"],
    "restaurants": ["rating", "rating_count", "cost"],
    "users": ["age", "family_size"],
    "reviews": ["rating", "review_date"],
}
for t, cols in checks.items():
    print("--", t)
    (spark.table(f"zomato_ai.silver.{t}")
         .agg(F.count("*").alias("total_rows"),
              *[F.sum(F.col(c).isNull().cast("int")).alias(f"{c}_nulls") for c in cols])
         .show(truncate=False))

-- orders
+----------+---------------------+----------------+--------------+------------------+-----------------------+
|total_rows|order_timestamp_nulls|order_date_nulls|subtotal_nulls|sales_amount_nulls|delivery_time_min_nulls|
+----------+---------------------+----------------+--------------+------------------+-----------------------+
|10000000  |0                    |0               |0             |0                 |849878                 |
+----------+---------------------+----------------+--------------+------------------+-----------------------+

-- restaurants
+----------+------------+------------------+----------+
|total_rows|rating_nulls|rating_count_nulls|cost_nulls|
+----------+------------+------------------+----------+
|148544    |87103       |87103             |134       |
+----------+------------+------------------+----------+

-- users
+----------+---------+-----------------+
|total_rows|age_nulls|family_size_nulls|
+----------+---------+-----------------+
|100000    

###Every order should have a real user and restaurant, and every order item a real order. All counts should be 0.

In [0]:
# Relationships
spark.sql("""SELECT count(*) AS orders_without_user FROM zomato_ai.silver.orders o
             LEFT ANTI JOIN zomato_ai.silver.users u ON o.user_id = u.user_id""").show()
spark.sql("""SELECT count(*) AS orders_without_restaurant FROM zomato_ai.silver.orders o
             LEFT ANTI JOIN zomato_ai.silver.restaurants r ON o.restaurant_id = r.restaurant_id""").show()
spark.sql("""SELECT count(*) AS items_without_order FROM zomato_ai.silver.order_items i
             LEFT ANTI JOIN zomato_ai.silver.orders o ON i.order_id = o.order_id""").show()



+-------------------+
|orders_without_user|
+-------------------+
|                  0|
+-------------------+

+-------------------------+
|orders_without_restaurant|
+-------------------------+
|                        0|
+-------------------------+

+-------------------+
|items_without_order|
+-------------------+
|                  0|
+-------------------+



In [0]:
# Delivery time sanity check (expect 849,878 nulls, 8 to 87 minutes)
spark.sql("""
SELECT count(*) AS total_rows,
       sum(CASE WHEN delivery_time_min IS NULL THEN 1 ELSE 0 END) AS delivery_time_nulls,
       min(delivery_time_min) AS min_minutes,
       max(delivery_time_min) AS max_minutes
FROM zomato_ai.silver.orders
""").show()

+----------+-------------------+-----------+-----------+
|total_rows|delivery_time_nulls|min_minutes|max_minutes|
+----------+-------------------+-----------+-----------+
|  10000000|             849878|          8|         87|
+----------+-------------------+-----------+-----------+

